In [ ]:
##initialization and imports
%load_ext autoreload
%autoreload 2

import sys
import os

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import ee
import geemap
import pandas as pd

from src.config import initialize_earth_engine, PRE_FIRE_START, PRE_FIRE_END, POST_FIRE_START, POST_FIRE_END
from src.data_loaders import get_susa_valley_aoi, get_bounding_box_extent, get_sentinel2_collection
from src.processors import calculate_ndvi, calculate_dnbr
from src.map_builder import create_base_map, add_sentinel2_layer, add_index_layer
from src.visualizers import plot_dnbr_histogram, plot_ndvi_scatterplot

initialize_earth_engine()

# 1. Load BOTH boundaries: CEMS irregular polygon (for science) & Bounding Box (for map framing)
cems_boundary = get_susa_valley_aoi()
aoi_bbox = get_bounding_box_extent()

print("[INFO] Setup complete with CEMS boundary and bounding box!")

In [ ]:
##Data Loading & Processing and clipping (Indices)
# Load image collections using the broader bounding box to capture context
pre_col = get_sentinel2_collection(aoi_bbox, PRE_FIRE_START, PRE_FIRE_END)
post_col = get_sentinel2_collection(aoi_bbox, POST_FIRE_START, POST_FIRE_END)

pre_fire_image = pre_col.median()
post_fire_image = post_col.median()

# Compute indices
pre_ndvi_clipped = calculate_ndvi(pre_fire_image).select('NDVI').clip(cems_boundary)
post_ndvi_clipped = calculate_ndvi(post_fire_image).select('NDVI').clip(cems_boundary)
dnbr_image_clipped = calculate_dnbr(pre_fire_image, post_fire_image).clip(cems_boundary)

print("[INFO] Rasters computed successfully!")

In [ ]:
# Interactive Map Visualization
Map = create_base_map()
Map.centerObject(aoi_bbox, 11)  # Center nicely using the broader box

# Add raster layers clipped/visualized
add_sentinel2_layer(Map, post_fire_image, name="Post-Fire True Color (S2)")

ndvi_palette = ['brown', 'yellow', 'lightgreen', 'green', 'darkgreen']
add_index_layer(Map, pre_ndvi_clipped, 'NDVI', ndvi_palette, -0.2, 0.8, 'Pre-Fire NDVI')
add_index_layer(Map, post_ndvi_clipped, 'NDVI', ndvi_palette, -0.2, 0.8, 'Post-Fire NDVI')

burn_palette = ['blue', 'gray', 'yellow', 'orange', 'red', 'purple']
add_index_layer(Map, dnbr_image_clipped, 'dNBR', burn_palette, -0.1, 0.6, 'Burn Severity (dNBR)')

# --- ADD VECTOR OVERLAYS ---
# 1. True CEMS Irregular Burn Scar (Red Outline)
Map.addLayer(cems_boundary, {'color': 'red', 'fillColor': '00000000', 'width': 2}, 'CEMS Official Burn Perimeter')

# 2. Study Bounding Box (Subtle Gray Frame)
Map.addLayer(aoi_bbox, {'color': 'gray', 'fillColor': '00000000', 'width': 1}, 'Study Bounding Box')

Map.addLayerControl()
Map

In [ ]:
left_layer = geemap.ee_tile_layer(
    pre_ndvi_clipped,
    {'min': -0.2, 'max': 0.8, 'palette': ndvi_palette},
    'Pre-Fire NDVI'
)
right_layer = geemap.ee_tile_layer(post_ndvi_clipped, {'min': -0.2, 'max': 0.8, 'palette': ndvi_palette}, 'Post-Fire NDVI')

split_map = geemap.Map(center=[45.167, 7.119], zoom=12)
split_map.split_map(left_layer=left_layer, right_layer=right_layer)

# Add CEMS boundary to split map
split_map.addLayer(cems_boundary, {'color': 'red', 'fillColor': '00000000', 'width': 2}, 'CEMS Burn Perimeter')
split_map